In [1]:
import os
import numpy as np
import time
import pandas as pd
from scipy import interpolate
import pickle
import sys
from astropy.io import ascii
from astropy.table import Table
from astropy import units as u
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, ListedColormap

In [2]:
def Enquiry(HashTable, InfoDict, Band1, Band2, dT1, dT2, dMag=None, Color=None):
    if abs(dT1) > abs(dT1-dT2):
        dT1, dT2 = dT1-dT2, -dT2    
    Ind1 = InfoDict['BandPairs'].index(Band1+Band2)
    #index for where 1st filter is, 2nd index for 2nd filter
    dT1grid = InfoDict['dT1s'][ abs( dT1 - InfoDict['dT1s'] ).argmin() ]
    dT2grid = InfoDict['dT2s'][ abs( dT2 - InfoDict['dT2s'] ).argmin() ]
    
    TimePairGrid = np.array([ InfoDict['dT1s'][ abs( dT1 - InfoDict['dT1s'] ).argmin() ], InfoDict['dT2s'][ abs( dT2 - InfoDict['dT2s'] ).argmin() ] ])
    # above will need some difference to set limit on difference in times
    Ind2 = np.where( (np.array(InfoDict['TimePairs']) == TimePairGrid ).all(axis=1) )[0][0]
    Results = HashTable[Ind1, Ind2]

    if dMag == None:
        pass        
    elif dMag<InfoDict['BinMag'][0] or dMag>=InfoDict['BinMag'][-1]:
        raise ValueError('The value of dMag is out of boundary, the available interval is [{:.2f}, {:.2f}).'.format(InfoDict['BinMag'][0], InfoDict['BinMag'][-1]))        
    else:
        Results = Results[np.where( dMag >= InfoDict['BinMag'] )[0][-1]]       

    if Color == None:
        pass        
    elif Color<InfoDict['BinColor'][0] or Color>=InfoDict['BinColor'][-1]:
        raise ValueError('The value of Color is out of boundary, the available interval is [{:.2f}, {:.2f}).'.format(InfoDict['BinColor'][0], InfoDict['BinColor'][-1]))
        
    else:
        Results = Results[..., np.where( Color >= InfoDict['BinColor'] )[0][-1] ]

    return Results

In [3]:
def loadCubeFile(FilePath):
    
    FileTime = FilePath[FilePath.find('Cube_')+4: FilePath.rfind('__')]
    
    with open(FilePath, 'rb') as f:
        InfoDict = pickle.load(f)
        print(InfoDict)
        HashTable = pickle.load(f)
    
    dT1Range = [InfoDict['dT1s'][0], InfoDict['dT1s'][-1]]
    dT2Range = [InfoDict['dT2s'][0], InfoDict['dT2s'][-1]]    
    dT1step = ( InfoDict['dT1s'][1:] - InfoDict['dT1s'][:-1] ).min()
    dT2step = ( InfoDict['dT2s'][1:] - InfoDict['dT2s'][:-1] ).min()
        
    StartObjNo = 'n/a'
    
    if 'StartObjNo' in InfoDict:
        StartObjNo = InfoDict['StartObjNo']
        
   # print('{:<35}ObjectNo: {:>5}, start at {:>3}. dT1 range = {}, step = {:>3}. dT2 range = {}, step = {:>3}. BandpairNo: {}.'.format(
   #     InfoDict['EventNames']+FileTime, InfoDict['ObjectNo'], StartObjNo, dT1Range, dT1step, dT2Range, dT2step, len(InfoDict['BandPairs'])))

    # InfoDict['OutliersRatio'] = InfoDict['Outliers'] / HashTable.sum()

    if 'Outliers' in InfoDict:
        print('\t{} outliers found, the ratio to the max value is {:.12f}.'.format(InfoDict['Outliers'], InfoDict['OutliersRatio']) )
        print('\tdMag range is {}, \n\tColor range is {}.'.format( InfoDict['dMagRange'], InfoDict['ColorRange'] ) )

    if 'Overflow' in InfoDict:
        print('\tData in the HashTable overflowed, the minimun value is {}.'.format(InfoDict['Overflow']))
        
    return InfoDict, HashTable;

In [4]:
def PlotSlice(HashTable1, InfoDict1, Band1, Band2, dT1, dT2, ax = None):
    # cut_gist_heat = ListedColormap(plt.colormaps["gist_heat"](np.linspace(0, 0.9, 256)))
    
    if ax == None: 
        fig, ax = plt.subplots(1,1)

    

    Map1 = Enquiry(HashTable1, InfoDict1, Band1, Band2, dT1, dT2) * 10000000

    ax.pcolor(InfoDict['BinMag'], InfoDict['BinColor'], np.transpose(Map1)+1,
                        norm=LogNorm(1, vmax=Map1.max()+1), cmap="gist_grey")

        # ax.scatter(Data[0], Data[1], c='mediumpurple', s=1, alpha=0.1, )


    ax.set_xlim([-1.5, 2])
    ax.set_ylim([-5, 8]) 


In [5]:
def PlotSliceWithTwoHists(HashTable1, InfoDict1, HashTable2, InfoDict2, Band1, Band2, dT1, dT2, ax = None):
    cut_gist_heat = ListedColormap(plt.colormaps["gist_heat"](np.linspace(0, 0.9, 256)))
    
    if ax == None: 
        fig, ax = plt.subplots(1,1)
    Map1 = Enquiry(HashTable1, InfoDict1, Band1, Band2, dT1, dT2) * 10000000
    Map2 = Enquiry(HashTable2, InfoDict2, Band1, Band2, dT1, dT2) * 10000000
    
    ax.pcolor(InfoDict['BinMag'], InfoDict['BinColor'], np.transpose(Map1)+1,
                       norm=LogNorm(1, vmax=Map1.max()+1), cmap='gist_gray')
    ax.pcolor(InfoDict2['BinMag'], InfoDict2['BinColor'], np.transpose(Map2),
                       norm=LogNorm(1, vmax=Map2.max()-1), cmap=cut_gist_heat)
            
    ax.set_xlim([-1.5, 2])
    ax.set_ylim([-5, 8]) 

In [6]:
dp1 = ascii.read("/lustre/lrspec/users/4300/cube/Data/Phot/DP1query4.ecsv")

In [7]:
dp1.info

<Table length=3194>
     name       dtype  unit                                                description                                               
-------------- ------- ---- ---------------------------------------------------------------------------------------------------------
      coord_ra float64  deg                                      Fiducial ICRS Right Ascension of centroid used for database indexing
     coord_dec float64  deg                                          Fiducial ICRS Declination of centroid used for database indexing
   diaObjectId   int64                                             Id of the DiaObject that this DiaForcedSource was associated with.
          band    str1                                            Abstract filter that is not associated with a particular instrument
       psfFlux float32  nJy                              Flux derived from linear least-squares fit of psf model forced on the calexp
    psfFluxErr float32  nJy           Unce

In [8]:
len(dp1[dp1["psfFlux"]<0])

31

In [9]:
dp1 = dp1[dp1["psfFlux"]>0]

In [10]:
len(np.unique(dp1["diaObjectId"]))

11

In [11]:
sel_objs = np.unique(dp1["diaObjectId"])[3:10]
dp1_sel = dp1[np.isin(dp1["diaObjectId"],sel_objs)]

In [12]:
pref_dT1s = np.arange(-480, 481, 15)
pref_dT2s = np.hstack(( np.arange(-1920, -1439, 30), np.arange(-480, 481, 30), np.arange(1440, 1921, 30) )) 
thrs = {'u': 23.9, 'g': 25.0, 'r': 24.7, 'i': 24.0, 'z': 23.3, 'y': 22.1}


In [13]:
obj_dfs = []

for n, obj in enumerate(np.unique(dp1_sel["diaObjectId"])):
    timepairs = []
    bandpairs = []
    dMags = []
    colors = []
    name = []
    dp1_obj = dp1_sel[dp1_sel["diaObjectId"] == obj]
    
    for i,mjd1 in enumerate(dp1_obj["expMidptMJD"]):
        dT1s2 = []
        dT2s2 = []
        dMs = []
        bps = []
        cs = []
        for j, mjd2 in enumerate(dp1_obj["expMidptMJD"]):
            if dp1_obj["band"][i] != dp1_obj["band"][j]:
                dT1 = -(mjd1-mjd2)*u.day
                dT1 = int((dT1.to(u.min)/u.min))

                
                if dT1 in pref_dT1s:
                    M1 = -2.5*np.log10(dp1_obj["psfFlux"][i])+31.4
                    M2 = -2.5*np.log10(dp1_obj["psfFlux"][j])+31.4
                    color = M1-M2
                    #= dp1_obj["psfFlux"][i] / dp1_obj["psfFlux"][j]
                    BandPair = dp1_obj["band"][i]+dp1_obj["band"][j]
                        
                
                    dT1s2.append(dT1)
                    cs.append(color)
                    bps.append(BandPair)
            
            else:
                dT2 = -(mjd1-mjd2)*u.day
                dT2 = int((dT2.to(u.min)/u.min))

                
                if dT2 in pref_dT2s and dT2 != 0:
                    M1 = -2.5*np.log10(dp1_obj["psfFlux"][i])+31.4
                    M2 = -2.5*np.log10(dp1_obj["psfFlux"][j])+31.4
                    
                    # dflux = (dp1_obj["psfFlux"][i] / dp1_obj["psfFlux"][j])

                    dT2s2.append(dT2)
                    #dMs.append((-2.5*np.log10(dflux))*np.sign(dT2))
                    dMs.append((M1-M2)*np.sign(dT2))
            
        for k, dT1 in enumerate(dT1s2):
            for h, dT2 in enumerate(dT2s2):
                timepairs.append((dT1,dT2))

                bandpairs.append(bps[k])
                colors.append(cs[k])
                
                dMags.append(dMs[h])

                name.append(obj)
                    

    obj_pts = pd.DataFrame()
    obj_pts["Timepairs"] = timepairs
    obj_pts["dMags"] = dMags
    obj_pts["Colors"] = colors
    obj_pts["Bandpairs"] = bandpairs
    obj_pts["Object Name"] = name

    obj_dfs.append(obj_pts)

    print(f'{n+1} of {len(np.unique(dp1_sel["diaObjectId"]))}')
print("Done!")

1 of 7
2 of 7
3 of 7
4 of 7
5 of 7
6 of 7
7 of 7
Done!


In [14]:
for obj in obj_dfs:
    print(obj["Object Name"])

0      609781520902651937
1      609781520902651937
2      609781520902651937
3      609781520902651937
4      609781520902651937
              ...        
113    609781520902651937
114    609781520902651937
115    609781520902651937
116    609781520902651937
117    609781520902651937
Name: Object Name, Length: 118, dtype: int64
0      609782208097419314
1      609782208097419314
2      609782208097419314
3      609782208097419314
4      609782208097419314
              ...        
261    609782208097419314
262    609782208097419314
263    609782208097419314
264    609782208097419314
265    609782208097419314
Name: Object Name, Length: 266, dtype: int64
0      609788942606139423
1      609788942606139423
2      609788942606139423
3      609788942606139423
4      609788942606139423
              ...        
112    609788942606139423
113    609788942606139423
114    609788942606139423
115    609788942606139423
116    609788942606139423
Name: Object Name, Length: 117, dtype: int64
0      

In [15]:
obj_dfs[4]["Timepairs"]

0      (15, 1440)
1        (15, 30)
2        (30, 60)
3     (30, -1680)
4     (15, -1710)
5     (15, -1710)
6     (15, -1440)
7       (15, -30)
8        (60, 30)
9        (60, 30)
10       (45, 30)
11       (45, 30)
12      (30, -30)
13      (15, -30)
14      (30, -30)
15      (15, -30)
16       (45, 30)
17     (-15, -30)
Name: Timepairs, dtype: object

In [16]:
u1 = set(obj_dfs[0]["Timepairs"]) & set(obj_dfs[1]["Timepairs"]) & set(obj_dfs[2]["Timepairs"]) & set(obj_dfs[3]["Timepairs"])&set(obj_dfs[5]["Timepairs"])&set(obj_dfs[6]["Timepairs"])

In [17]:
list(u1)

[(15, 30), (-30, -60), (15, 1440), (-15, -30), (45, 30), (15, -30)]

In [18]:
list(u1)[-4]

(15, 1440)

In [19]:
for i in range(len(obj_dfs)):
    print(obj_dfs[i]["Object Name"][0],'\n',obj_dfs[i]["Bandpairs"][obj_dfs[i]["Timepairs"]==(15,1440)])

609781520902651937 
 1     zg
2     zg
27    zr
29    zr
Name: Bandpairs, dtype: object
609782208097419314 
 75     zg
80     zr
113    zr
114    zr
116    zr
117    zr
118    zr
119    zr
145    zg
146    zg
147    zr
163    ri
166    ri
176    rg
Name: Bandpairs, dtype: object
609788942606139423 
 26     zr
43     ri
46     ri
106    zr
107    zr
109    zr
110    zr
Name: Bandpairs, dtype: object
611253629533290657 
 87     rg
88     rg
100    zr
118    zr
119    zr
121    zr
122    zr
125    zr
153    ri
156    ri
161    ri
165    ri
167    ri
173    rg
174    rg
194    rg
205    zg
234    zr
237    zr
240    zr
242    zr
244    zg
Name: Bandpairs, dtype: object
611255210081255575 
 0    zg
Name: Bandpairs, dtype: object
611255759837069401 
 4      zg
71     zr
72     zr
73     zr
74     zr
75     zr
76     zr
77     zr
81     ri
119    rg
133    zg
Name: Bandpairs, dtype: object
611256447031836769 
 22    zr
23    zr
24    zr
25    zr
26    zr
28    zr
Name: Bandpairs, dtype: objec

In [20]:
EventNames = ['AGN', 'CART', 'EB', 'ILOT', 'KN_B19', 'KN_K17', 'MIRA', 'Mdwarf',
              'PISN', 'RRL', 'SLSN-I', 'SNII-NMF', 'SNII-Templates', 'SNIIn',
              'SNIa-91bg', 'SNIa-SALT2', 'SNIax', 'SNIbc-MOSFIT',
              'SNIbc-Templates', 'TDE', 'V19_CC+HostXT', 'uLens-Binary',
              'uLens-Single-GenLens', 
              #'uLens-Single_PyLIMA'
             ]


In [21]:
PathCubeFolder = '/lustre/lrspec/users/4300/cube/Data/Datacube/All_Events'
CubeFileNames = os.listdir(PathCubeFolder)
CubeFileNames = [ ii for ii in CubeFileNames if '.pkl' in ii ]

In [35]:
mp_colors = ["red", "orange", "yellow", "green","pink", "blue", "violet"]
mp_shapes = ["s","D","*","^","<","v","P"]
sizes = [7, 7, 11, 8, 8, 8, 9]
cut_gist_heat = ListedColormap(plt.colormaps["gist_heat"](np.linspace(0, 0.9, 256)))

In [46]:
bandpairs = ["rg","zr"]
timepairs = [(45,1440),(15,1440)]
# Band1 = 'r'
# Band2 = 'g'
# dT1 = 45
# dT2 = 1440


Space = 0.05
ColNo = 3
RowNo = 7

fig, axs = plt.subplots(RowNo, ColNo, figsize = (18, 4*RowNo), sharex=True, sharey=True, dpi = 750)
fig.subplots_adjust(hspace=Space, wspace=Space)
axsflat = axs.flatten()

q = 1
ii=0

for EventName in EventNames:
    
    print('|', end='')

    if EventName == 'V19_CC+HostXT'or 'KN_K' in EventName:
        continue
    for CubeFileName in CubeFileNames:
        if EventName in CubeFileName:
            break
            
    CubeFilePath = os.path.join(PathCubeFolder, CubeFileName)

    with open(CubeFilePath, 'rb') as f:
        InfoDict = pickle.load(f)
        Cube = pickle.load(f)
        
    CubeNorm = Cube / Cube.max(-1, keepdims=True).max(-2, keepdims=True)
    np.nan_to_num(CubeNorm, copy=False);
    
    Map = Enquiry(CubeNorm, InfoDict, bandpairs[q][0], bandpairs[q][1], timepairs[q][0], timepairs[q][1])    
    
    Map = Map*100000

    
    axsflat[ii].pcolor(InfoDict['BinMag'], InfoDict['BinColor'], np.transpose(Map)+1,
                      norm=LogNorm(1, vmax=Map.max()+1), cmap='gist_gray')

    if EventName == 'SNIa-SALT2':
        CAATPath = "/lustre/lrspec/users/4300/cube/Data/Datacube/CAAT/SweetSpot/ProbCube_0626_1050__Neg_CAATSNIa.pkl"
        with open(CAATPath, 'rb') as f:
            InfoDict2 = pickle.load(f)
            Cube2 = pickle.load(f)
                
        CubeNorm2 = Cube2 / Cube2.max(-1, keepdims=True).max(-2, keepdims=True)
        np.nan_to_num(CubeNorm2, copy=False);
        
        Map2 = Enquiry(CubeNorm2, InfoDict2, bandpairs[q][0], bandpairs[q][1], timepairs[q][0], timepairs[q][1])    
        
        Map2 = Map2*100000
        
        axsflat[ii].pcolor(InfoDict2['BinMag'], InfoDict2['BinColor'], np.transpose(Map2),
                   norm=LogNorm(1, vmax=Map2.max()-1), cmap=cut_gist_heat, alpha = 0.7)

    for k, obj_pts in enumerate(obj_dfs):
        if k != 4:
            axsflat[ii].plot(list(obj_pts["dMags"][obj_pts["Bandpairs"]== bandpairs[q]][obj_pts["Timepairs"]==timepairs[q]])[0],
                           list(obj_pts["Colors"][obj_pts["Bandpairs"]== bandpairs[q]][obj_pts["Timepairs"]==timepairs[q]])[0], 
                           mp_shapes[k], mec = "black", 
                             mew = 0.7, color = "#00dcbbff", ms = sizes[k])


    
    axsflat[ii].text(0.95, 0.95, EventName, c='w', ha='right', va='top', transform=axsflat[ii].transAxes, size = "x-large")    
    if ii%3 == 0:
        axsflat[ii].set_ylabel(f"{bandpairs[q][0]} - {bandpairs[q][1]}", size = "x-large")
    if ii in [18,19,20]:
        axsflat[ii].set_xlabel(f"$\\Delta$ {bandpairs[q][0]}", size = "x-large")
        
    ii += 1
        
axs[0,0].set_xlim([-1.5, 2])
axs[0,0].set_ylim([-5, 5])

#fig.suptitle(f"$\\Delta T_1$ = {timepairs[q][0]} min, $\\Delta T_2$ = {timepairs[q][1]} min", x = 0.4, y = 0.8, size = 10)
title = f"$\\Delta T_1$ = {timepairs[q][0]} min, $\\Delta T_2$ = {timepairs[q][1]} min"
fig.suptitle(title, y = 0.89, size ="xx-large", color = "black")
plt.savefig("/lustre/lrspec/users/4300/cube/Figures/DP1_comparison_151440_60360_zr")

|||||||||||||||||||||||

In [ ]:
list(obj_dfs[1]["dMags"][obj_dfs[1]["Bandpairs"]== bandpairs[q]][obj_dfs[1]["Timepairs"]==timepairs[q]])[0]